In [6]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

weather_data = pd.read_csv('/kaggle/input/datasets/micahluftig/dasaxz/other_environmental_variables.csv')
print(weather_data.columns)
print(weather_data[['apt_seen', 'p_deceased']].describe())

Index(['date', 'temp(°F)', 'feels_like(°F)', 'humidity(%)', 'precip(in)',
       'snow(in)', 'snow_depth(in)', 'wind_gust(mph)', 'wind_speed(mph)',
       'sea_level_pressure(mb)', 'visibility(mi)', 'solar_radiation(W/m²)',
       'solar_energy(MJ/m²)', 'uv_index', 'apt_seen', 'p_deceased'],
      dtype='object')
          apt_seen   p_deceased
count  2708.000000  2708.000000
mean     43.161743     1.729321
std      17.233797     1.794029
min       1.000000     0.000000
25%      33.000000     0.000000
50%      42.000000     1.000000
75%      53.000000     3.000000
max     137.000000    14.000000


In [7]:
# Standardize both predictors (learned from the intake regression's convergence
# failure earlier — pressure's raw scale vs. small death counts causes the
# same MLE instability problem here)
weather_data['pressure_z'] = (weather_data['sea_level_pressure(mb)'] - weather_data['sea_level_pressure(mb)'].mean()) / weather_data['sea_level_pressure(mb)'].std()
weather_data['feels_like_z'] = (weather_data['feels_like(°F)'] - weather_data['feels_like(°F)'].mean()) / weather_data['feels_like(°F)'].std()

model_df = weather_data.dropna(subset=['pressure_z', 'feels_like_z', 'p_deceased'])

X = model_df[['pressure_z', 'feels_like_z']]
X = sm.add_constant(X)
y = model_df['p_deceased']

nb_model = sm.NegativeBinomial(y, X)
nb_result = nb_model.fit(method='bfgs', maxiter=200)
print(nb_result.summary())

Optimization terminated successfully.
         Current function value: 1.760936
         Iterations: 9
         Function evaluations: 10
         Gradient evaluations: 10
                     NegativeBinomial Regression Results                      
Dep. Variable:             p_deceased   No. Observations:                 2708
Model:               NegativeBinomial   Df Residuals:                     2705
Method:                           MLE   Df Model:                            2
Date:                Mon, 20 Jul 2026   Pseudo R-squ.:                0.003010
Time:                        20:51:16   Log-Likelihood:                -4768.6
converged:                       True   LL-Null:                       -4783.0
Covariance Type:            nonrobust   LLR p-value:                 5.582e-07
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.5420      0.020 

In [8]:
# Convert both standardized coefficients back to interpretable %-per-unit effects
pressure_std = weather_data['sea_level_pressure(mb)'].std()
feels_like_std = weather_data['feels_like(°F)'].std()

coef_pressure = nb_result.params['pressure_z']
coef_feels_like = nb_result.params['feels_like_z']

pct_change_per_mb_mortality = (np.exp(-coef_pressure / pressure_std) - 1) * 100
pct_change_per_degf_mortality = (np.exp(-coef_feels_like / feels_like_std) - 1) * 100

print(f"% change in expected deaths per 1 mb pressure drop: {pct_change_per_mb_mortality:.3f}%")
print(f"% change in expected deaths per 1°F temperature drop: {pct_change_per_degf_mortality:.3f}%")
print(f"Mean daily deaths: {model_df['p_deceased'].mean():.3f}")
print(f"Variance: {model_df['p_deceased'].var():.3f}")
print(f"Dispersion ratio (var/mean): {model_df['p_deceased'].var() / model_df['p_deceased'].mean():.3f}")

% change in expected deaths per 1 mb pressure drop: 0.372%
% change in expected deaths per 1°F temperature drop: -0.575%
Mean daily deaths: 1.729
Variance: 3.219
Dispersion ratio (var/mean): 1.861


In [9]:
weather_data['feels_like_z'] = (weather_data['feels_like(°F)'] - weather_data['feels_like(°F)'].mean()) / weather_data['feels_like(°F)'].std()

model_df_intake = weather_data.dropna(subset=['pressure_z', 'feels_like_z', 'apt_seen'])

X = model_df_intake[['pressure_z', 'feels_like_z']]
X = sm.add_constant(X)
y = model_df_intake['apt_seen']

nb_model_intake_multi = sm.NegativeBinomial(y, X)
nb_result_intake_multi = nb_model_intake_multi.fit(method='bfgs', maxiter=200)
print(nb_result_intake_multi.summary())

Optimization terminated successfully.
         Current function value: 4.270448
         Iterations: 11
         Function evaluations: 14
         Gradient evaluations: 14
                     NegativeBinomial Regression Results                      
Dep. Variable:               apt_seen   No. Observations:                 2708
Model:               NegativeBinomial   Df Residuals:                     2705
Method:                           MLE   Df Model:                            2
Date:                Mon, 20 Jul 2026   Pseudo R-squ.:                0.009297
Time:                        20:51:16   Log-Likelihood:                -11564.
converged:                       True   LL-Null:                       -11673.
Covariance Type:            nonrobust   LLR p-value:                 7.358e-48
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            3.7576      0.008

In [10]:
X = model_df[['feels_like_z']]
X = sm.add_constant(X)
y = model_df['p_deceased']

nb_model_temp_only = sm.NegativeBinomial(y, X)
nb_result_temp_only = nb_model_temp_only.fit(method='bfgs', maxiter=200)
print(nb_result_temp_only.summary())

feels_like_std = weather_data['feels_like(°F)'].std()
coef_feels_like = nb_result_temp_only.params['feels_like_z']
pct_change_per_degf = (np.exp(-coef_feels_like / feels_like_std) - 1) * 100
print(f"% change in expected deaths per 1°F temperature drop: {pct_change_per_degf:.3f}%")
print(f"Mean daily deaths: {model_df['p_deceased'].mean():.3f}")
print(f"Variance: {model_df['p_deceased'].var():.3f}")

Optimization terminated successfully.
         Current function value: 1.761061
         Iterations: 7
         Function evaluations: 8
         Gradient evaluations: 8
                     NegativeBinomial Regression Results                      
Dep. Variable:             p_deceased   No. Observations:                 2708
Model:               NegativeBinomial   Df Residuals:                     2706
Method:                           MLE   Df Model:                            1
Date:                Mon, 20 Jul 2026   Pseudo R-squ.:                0.002940
Time:                        20:51:16   Log-Likelihood:                -4769.0
converged:                       True   LL-Null:                       -4783.0
Covariance Type:            nonrobust   LLR p-value:                 1.138e-07
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.5421      0.020   